# 01 — Exploratory Data Analysis

**Dataset**: Netflix Movies & TV Shows (till 2025), split into two CSV files — `netflix_movies_detailed_up_to_2025.csv` and `netflix_tv_shows_detailed_up_to_2025.csv`.

This notebook profiles the raw schema, analyses missing values, produces five visualisations, and documents the two research hypotheses that drive the model design.

In [ ]:
import sys
from pathlib import Path

# Make the src/ package importable from a notebooks/ directory.
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import NetflixDataLoader

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 40)

## 2.1 Data Schema

In [ ]:
DATA_DIR = PROJECT_ROOT / 'data'
MOVIES_CSV = DATA_DIR / 'netflix_movies_detailed_up_to_2025.csv'
TV_CSV = DATA_DIR / 'netflix_tv_shows_detailed_up_to_2025.csv'

loader = NetflixDataLoader([MOVIES_CSV, TV_CSV])
df_raw = loader.load()
print('Shape:', df_raw.shape)
print('Columns:', df_raw.columns.tolist())
df_raw.info()

In [ ]:
df_raw.describe(include='all').transpose()

In [ ]:
df_raw.head()

## 2.2 Missing Values Analysis

In [ ]:
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw)) * 100
missing_report = (
    pd.DataFrame({'missing': missing, 'missing_pct': missing_pct.round(2)})
    .sort_values('missing_pct', ascending=False)
)
missing_report

In [ ]:
df = loader.clean(df_raw)
print('After cleaning:', df.shape)
df.head()

## 2.3 Key Observations

Note: in this dataset `rating` is a **numeric TMDB score** (0-10), not an MPAA rating. Movies have no `duration` (100% null) while TV shows use the form `"N Seasons"`.

In [ ]:
print('Type distribution:')
print(df['type'].value_counts())
print()
print('Rating (TMDB score) summary:')
print(df['rating'].describe())
print()
print('Release year range:', int(df['release_year'].min()), '->', int(df['release_year'].max()))
print()
first_country = df['country'].str.split(',').str[0].str.strip()
print('Country distribution (top 10):')
print(first_country.value_counts().head(10))
print()
print('Language distribution (top 10):')
print(df['language'].value_counts().head(10))

In [ ]:
all_genres = pd.Series([g for row in df['genres'] for g in row])
print('Top 15 genres:')
print(all_genres.value_counts().head(15))

## 2.4 Visualisations

In [ ]:
# Chart 1: content type distribution
fig, ax = plt.subplots(figsize=(6, 4))
df['type'].value_counts().plot(kind='bar', ax=ax, color=['#E50914', '#221F1F'])
ax.set_title('Netflix Catalog: Movies vs TV Shows')
ax.set_xlabel('Content type')
ax.set_ylabel('Number of titles')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 2: release-year histogram
fig, ax = plt.subplots(figsize=(9, 4))
df['release_year'].dropna().astype(int).plot(kind='hist', bins=40, ax=ax, color='#E50914')
ax.set_title('Distribution of Release Year')
ax.set_xlabel('Release year')
ax.set_ylabel('Number of titles')
plt.tight_layout()
plt.show()

In [ ]:
# Chart 3: top 10 genres
fig, ax = plt.subplots(figsize=(9, 5))
all_genres.value_counts().head(10)[::-1].plot(kind='barh', ax=ax, color='#221F1F')
ax.set_title('Top 10 Genres on Netflix')
ax.set_xlabel('Number of titles')
ax.set_ylabel('Genre')
plt.tight_layout()
plt.show()

In [ ]:
# Chart 4: missing values heatmap
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(df_raw.isnull(), cbar=False, yticklabels=False, ax=ax, cmap='Reds')
ax.set_title('Missing Values Heatmap (raw data)')
plt.tight_layout()
plt.show()

In [ ]:
# Chart 5: titles added per year
fig, ax = plt.subplots(figsize=(9, 4))
added = df['year_added'].dropna().astype(int).value_counts().sort_index()
added.plot(kind='line', marker='o', ax=ax, color='#E50914')
ax.set_title('Titles Added to Netflix Per Year')
ax.set_xlabel('Year added')
ax.set_ylabel('Number of titles')
plt.tight_layout()
plt.show()

## 3. Research Design — Two Hypotheses

### Hypothesis 1 — Content-Based Similarity
> Users who watch content with similar genres, cast, and director characteristics will prefer recommendations from those clusters over random recommendations.

- **Why it matters**: Content metadata is the richest available signal in this dataset; testing similarity-based retrieval validates whether latent features are informative.
- **How to test**: Build content embeddings → compute cosine similarity → compare top-K recommended items against a genre-overlap relevance proxy and against random recommendations.
- **Validation metric**: Precision@10, Recall@10, NDCG@10, and lift of model precision over random-pair genre overlap.

### Hypothesis 2 — Temporal Recency Bias
> Recently added content (within 2 years of viewing) will appear disproportionately in high-engagement recommendation lists compared to older content.

- **Why it matters**: Streaming platforms prioritise catalog freshness; if the model reflects this, it mirrors real-world production behaviour.
- **How to test**: Bin items by `date_added` year → compare the mean `release_year` of recommended items against the catalog average, and compute the time-bias score.
- **Validation metric**: Mean release year of recommended items, `time_bias_score = mean(year_recommended) / mean(year_catalog)`.

Both hypotheses are validated in notebook 04.